In [1]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [2]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

c:\Users\K B\Desktop\Regression_ML_EndtoEnd\.venv\Scripts\python.exe
3.0.4
c:\Users\K B\Desktop\Regression_ML_EndtoEnd\.venv\Lib\site-packages\xgboost\__init__.py


In [3]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

c:\Users\K B\Desktop\Regression_ML_EndtoEnd\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv("C:\\Users\\K B\\Desktop\\Regression_ML_EndtoEnd\\data\\processed\\feature_engineered_train.csv")
eval_df = pd.read_csv("C:\\Users\\K B\\Desktop\\Regression_ML_EndtoEnd\\data\\processed\\feature_engineered_eval.csv")

# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [5]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [8]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================

from pathlib import Path

# Create a proper file:// URI for the mlruns folder
mlruns_path = Path.cwd() / "mlruns"

mlflow.set_tracking_uri(mlruns_path.as_uri())
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

2026/09/30 22:14:48 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.
[I 2026-09-30 22:14:48,311] A new study created in memory with name: no-name-cfdd750c-46fe-4a08-b1fa-9bd2a75a07ac
[I 2026-09-30 22:16:04,997] Trial 0 finished with value: 70923.9809457346 and parameters: {'n_estimators': 714, 'max_depth': 8, 'learning_rate': 0.08506952917602721, 'subsample': 0.526156915442229, 'colsample_bytree': 0.5185452152533419, 'min_child_weight': 6, 'gamma': 0.06545864812731339, 'reg_alpha': 1.223239313512472e-08, 'reg_lambda': 7.001228953648629e-05}. Best is trial 0 with value: 70923.9809457346.
[I 2026-09-30 22:17:03,640] Trial 1 finished with value: 73404.72588594577 and parameters: {'n_estimators': 735, 'max_depth': 5, 'learning_rate': 0.1449456679827409, 'subsample': 0.9859013044123235, 'colsample_bytree': 0.6769490079758088, 'min_child_weight': 3, 'gamma': 2.7825409585134784, 'reg_alpha': 4.037366420144808e-08, 'reg_lambd

Best params: {'n_estimators': 796, 'max_depth': 5, 'learning_rate': 0.07915334176996146, 'subsample': 0.7194016553757645, 'colsample_bytree': 0.575977925299366, 'min_child_weight': 5, 'gamma': 0.8026949061901972, 'reg_alpha': 0.0010578744192599376, 'reg_lambda': 5.049412466767441}


In [9]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model") # type: ignore

Final tuned model performance:
MAE: 32433.765234200815
RMSE: 70301.96570758411
R²: 0.9618060154480947


c:\Users\K B\Desktop\Regression_ML_EndtoEnd\.venv\Lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [22:34:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2026/09/30 22:34:47 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/09/30 22:34:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
